In [1]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

Current Python version: 3.10.12 (main, May 27 2025, 17:12:29) [GCC 11.4.0]


In [2]:
from pathlib import Path
import librosa
import numpy as np
import time
import optuna
from optuna.trial import Trial
import joblib
import random

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict

In [4]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [5]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/train/"
STUDY = "/workspaces/dev/study/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [6]:
src = Path(SOURCE)
study_path = Path(STUDY) / "study.pkl"
study_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
src_folder = []

for dirpath in (p for p in src.rglob("*") if p.is_dir()):
    trans_files = list(dirpath.glob("*.trans.txt"))
    if len(trans_files) == 0:
        continue
    elif len(trans_files) > 1:
        print(f"Multiple transcription files found in {dirpath}, skipping.")
        continue
    src_folder.append(dirpath)

In [ ]:
def objective(trial:Trial):
    hyperparameters = SafetyDict({
        "weighted_and_offset_token_boundary": trial.suggest_int(
            "weighted_and_offset_token_boundary", 4000, 16000, step=100
        ),
        "duration_filter_z":{
            "default": 2.0,
            "en": trial.suggest_float("duration_filter_z", 3.0, 5.0, step = 0.1)
        },
        "probability_filter":{
            "z":{
                "default": 3.0,
                "en": trial.suggest_float("probability_filter_z", 3.0, 5.0, step = 0.1)
            },
            "min_prob": {
                "default": 1.0,
                "en": trial.suggest_float("probability_filter_min_prob", 0, 1, step = 0.05)
            },
        },
        "selector":{
            "search_range_sc": {
                "default": 24000,
                "en": trial.suggest_int("search_range_sc", 0, 48000, step=1000)
            },
            "threshold":{
                "default": 0.5,
                "en": trial.suggest_float("threshold", 0, 1, step=0.05)
            },
            "padding": {
                "default": 3200,
                "en": trial.suggest_int("padding", 0, 18000, step=100)
            },
            "tolerance": {
                "default": 8000,
                "en": trial.suggest_int("tolerance", 0, 16000, step=100)
            }
        },
        "max_overlap_duration": 96000
    })

    token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter=hyperparameters)

    def transcriber(flac:Path) -> TRNFormat:
        audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

        completed = []
        param = Param()
        # start_time = time.perf_counter()
        for segment in segment_audio(audio):
            param.chunk = segment
            param.language="en"
            result:Result = token_streamer.process(param)
            completed.extend(result.completed)
            param.update(result)
        completed.extend(result.candidate)
        # end_time = time.perf_counter()
        # print(f"Processed {flac.stem} in {end_time - start_time:.2f} seconds")

        return TRNFormat(
            id = flac.stem,
            text = normalize_text_only_en(
                " ".join([s.text for s in completed])
            ).upper()
        )

    samples = random.sample(src_folder, 1)

    data = {}
    for sample in samples:
        trans_txt = next(sample.glob("*.trans.txt"))
        ref = trans_txt_to_sclite_trn(trans_txt)
        hyp = [transcriber(flac) for flac in sorted(sample.glob("*.flac"))]
        data[sample.stem] = {
            "ref": ref,
            "hyp": hyp
        }

    concat_result = {}
    for value in data.values():
        for k, v in value.items():
            if k not in concat_result:
                concat_result[k] = []
            concat_result[k].extend(v)

    output = sclite_trn(concat_result["ref"], concat_result["hyp"])
    result = parse_sclite_summary(output)

    return result["wer_percent"]

In [ ]:
if study_path.exists():
    study = joblib.load(study_path)
else:
    study = optuna.create_study(direction="minimize")

In [ ]:
for _ in range(1):
    study.optimize(objective, n_trials=5)
    joblib.dump(study, study_path)

[I 2025-07-04 14:44:38,796] Trial 2 finished with value: 7.8 and parameters: {'weighted_and_offset_token_boundary': 12800, 'duration_filter_z': 3.9, 'probability_filter_z': 3.9, 'probability_filter_min_prob': 0.35000000000000003, 'search_range_sc': 14000, 'threshold': 0.05, 'padding': 11500, 'tolerance': 9800}. Best is trial 2 with value: 7.8.
[W 2025-07-04 14:52:04,471] Trial 3 failed with parameters: {'weighted_and_offset_token_boundary': 4200, 'duration_filter_z': 4.6, 'probability_filter_z': 3.0, 'probability_filter_min_prob': 0.8, 'search_range_sc': 44000, 'threshold': 0.8, 'padding': 3000, 'tolerance': 10100} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_59409/3748247978.py", line 72, in objective
    hyp = [transcriber(flac) for flac in sorted(sample.glob("*.flac"))]
  File "/tmp/ipy

KeyboardInterrupt: 

In [ ]:
study.best_value

In [ ]:
study.best_params